### 프로덕션 레벨 ReAct 워크플로우 구축 (LangGraph)
- LangGraph를 활용하여 상태(State) 기반의 에이전트 워크플로우를 설계한다.
- 순환(Cyclic) 그래프 구조를 통해 모델이 원하는 결과를 낼 때까지 스스로 도구를 호출하고 검증하는 로직을 구현한다.

### 1. 상태(State) 및 도구 정의
LangGraph는 각 노드(Node) 간에 전달되는 상태(State)를 정의해야 합니다. 에이전트의 상태는 주고받은 메시지들의 목록입니다.

In [ ]:
import os
import operator
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage,ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [ ]:
# 1. 상태객체(state object) 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. 도구 정의
@tool
def get_current_time(loaction:str)->str:
    '''특정 지역의 현재 시간을 반환합니다.'''
    import datetime
    return f"{loaction}의 현재 시간은 {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} 입니다"

tools = [get_current_time]
# langgraph.prebuilt 의 ToolNode 를 사용하면 실행노드를 쉽게 만들수 있음
tool_node = ToolNode(tools)
print('상태 정의 및 도구 초기화 완료')

### 2. 에이전트 노드 및 라우팅 함수 구현
모델을 호출하는 노드(Node)와, 모델의 응답에 따라 도구를 실행할지 종료할지 결정하는 엣지(Edge) 로직을 만듭니다.

In [ ]:
# 1. 모델과 도구 바인딩(Function calling)
model = ChatOpenAI(api_key=os.getenv('OPENAI_API_KEY'),model='gpt-5.4-nano',temperature=0)
model_with_tools = model.bind_tools(tools)

# 2. 에이전트 노드 함수 : 상태를 받아 모델을 호출하고 상태를 업데이트
def call_model(state:AgentState):
    messages = state['messages']
    response = model_with_tools.invoke(messages)
    return {'messages':[response]}

# 3.라우팅 로직 : 모델이 도구 호출(tool_calls)을 요청했는지 확인
def should_continue(state:AgentState):
    messages = state['messages']
    last_message = messages[-1]
    # 마지막 메세지에 도구 호출 정보가 있다면 도구 노드로 이동
    if last_message.tool_calls:
        return 'continue'
    # 도구 호출이 없다면(최종답변을 도출했다면) 종료
    return 'end'

print('에이전트 노드와 라우터 준비')